# Generate Kaggle Submission
Format predictions to match sample_submission.csv


In [1]:
# Pre-load CUDA 13.x shared libs so bitsandbytes finds them regardless of LD_LIBRARY_PATH
import ctypes, os as _os
_cu13 = _os.path.join(_os.path.dirname(_os.__file__),
    'site-packages/nvidia/cu13/lib')
for _lib in ['libcudart.so.13', 'libcublas.so.13', 'libcublasLt.so.13', 'libnvJitLink.so.13']:
    try: ctypes.CDLL(_os.path.join(_cu13, _lib))
    except OSError: pass
del _cu13, _lib

# === ENVIRONMENT & FILEPATH SETUP ===
import os
import sys

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)

# Clear src modules to allow reloading
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

DATA_DIR = f"{codebase_path}data"
CACHE_DIR = f"{codebase_path}output/cache"
MODELS_DIR = f"{codebase_path}output/models"
ARTIFACTS_DIR = f"{codebase_path}artifacts"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

models = ['Model_A', 'Model_B', 'Model_C', 'Model_D', 'Model_E', 'Model_F', 'Model_G', 'Model_H', 'Model_I', 'Model_J', 'Model_K']
perf_cols = [f"{m}_performance" for m in models]
cost_cols = [f"{m}_cost" for m in models]
max_cost_per_query = train[cost_cols].max(axis=1)
global_avg_max_cost = max_cost_per_query.mean()

rewards = pd.DataFrame(index=train.index, columns=models)
for m in models:
    p = train[f"{m}_performance"]
    c = train[f"{m}_cost"]
    rewards[m] = 0.85 * p - 0.15 * (c / global_avg_max_cost)
y_reg = rewards.values

import torch
import gc
from sentence_transformers import SentenceTransformer
from transformers import BitsAndBytesConfig
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

# Load or compute Train Features
train_feat_path = f"{CACHE_DIR}/dense_Qwen-Qwen3-Embedding-8B_features.npy"
if os.path.exists(train_feat_path):
    print("Loading cached train features...")
    X_train_full = np.load(train_feat_path)
else:
    print("Computing train features...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
    )
    model = SentenceTransformer('Qwen/Qwen3-Embedding-8B', 
                                model_kwargs={"quantization_config": bnb_config, "attn_implementation": "sdpa"})
    model.max_seq_length = 2048
    X_train_full = model.encode(train['query'].tolist(), show_progress_bar=True, batch_size=4)
    np.save(train_feat_path, X_train_full)
    del model
    torch.cuda.empty_cache()
    gc.collect()

# Compute Test Features
print("Computing test features...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
model = SentenceTransformer('Qwen/Qwen3-Embedding-8B', 
                            model_kwargs={"quantization_config": bnb_config, "attn_implementation": "sdpa"})
model.max_seq_length = 2048
X_test = model.encode(test['query'].tolist(), show_progress_bar=True, batch_size=4)
del model
torch.cuda.empty_cache()
gc.collect()

# 10 K-Fold Ridge Regression
K_FOLDS = 10
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
test_pred_rewards = np.zeros((len(test), len(models)))

for train_idx, val_idx in kf.split(X_train_full):
    X_train_fold = X_train_full[train_idx]
    for m_idx in range(len(models)):
        y_train_fold = y_reg[train_idx, m_idx]
        reg = Ridge(alpha=1.0)
        reg.fit(X_train_fold, y_train_fold)
        test_pred_rewards[:, m_idx] += reg.predict(X_test) / K_FOLDS

test_preds = np.argmax(test_pred_rewards, axis=1)
test['pred_model'] = [models[int(p)] for p in test_preds]


Loading cached train features...
Computing test features...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Batches:   0%|          | 0/638 [00:00<?, ?it/s]

In [4]:
# Verify submission format matches sample_sub exactly
submission = test[['ID', 'pred_model']]
assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)

# Save to output
output_file = f"{codebase_path}output/submissions/ridge_regression_dense_Qwen-Qwen3-Embedding-8B_10kfold.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
submission.to_csv(output_file, index=False)
print(f"Saved to {output_file}")


Saved to ../output/submissions/ridge_regression_dense_Qwen-Qwen3-Embedding-8B_10kfold.csv
